In [ ]:
import numpy as np
import pandas as pd
import time
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

# Configs
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")
np.random.seed(42)



SEED = 42
np.random.seed(SEED)

In [ ]:
# Carregamento e pré-processamento dos dados
df = pd.read_excel("datasets/Dry_Bean_Dataset.xlsx")

print(f"\nBase carregada: {df.shape[0]} instâncias, {df.shape[1]} colunas")
print(df.head())

# Separação entre features e target
X = df.drop("Class", axis=1).values
y = df["Class"].values

# Codificando os rótulos para números
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
classes = label_encoder.classes_
print(f"\nClasses: {classes}")
print(f"Mapeamento: {dict(zip(range(len(classes)), classes))}")

# Normalização Min-Max
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nDados pré-processados:")
print(f"Features: {X_scaled.shape[1]} atributos numéricos")
print(f"Classes: {len(classes)}")

In [ ]:
# Divisão dos Dados
print("\n" + "="*60)
print("DIVISÃO DOS DADOS (70% treino, 30% teste)")
print("="*60)

# Divisão única: 70% treino, 30% teste
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, 
    test_size=0.3,           # 30% para teste
    random_state=SEED, 
    stratify=y_encoded       # Mantém proporção das classes
)

print(f"Conjunto de Treino: {X_train.shape[0]} instâncias ({100*X_train.shape[0]/len(X_scaled):.1f}%)")
print(f"Conjunto de Teste:  {X_test.shape[0]} instâncias ({100*X_test.shape[0]/len(X_scaled):.1f}%)")

# Verificar estratificação
print("\nDistribuição das classes no treino:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {classes[u]}: {c} ({100*c/len(y_train):.1f}%)")

In [ ]:
# Questão 1: Exploração Inicial
print("\n" + "="*60)
print("Questão 1: Exploração Inicial")
print("5 treinamentos com diferentes inicializações")
print("="*60)

# Configuração experimental
HIDDEN_LAYERS = (64, 32) # 2 camadas ocultas com 64 e 32 neurônios
ACTIVATION = "relu" # ReLU para camadas ocultas
OUTPUT_ACTIVATION = "softmax"
LOSS = "cross_entropy"
LEARNING_RATE_INIT = 0.001
MAX_ITER = 200
BATCH_SIZE = 32
SOLVER = "adam"

print(f"\nConfiguração experimental:")
print(f" - Arquitetura: {HIDDEN_LAYERS}")
print(f" - Função ativação oculta: {ACTIVATION}")
print(f" - Função ativação saída: {OUTPUT_ACTIVATION}")
print(f" - Função perda: {LOSS}")
print(f" - Otimizador: {SOLVER}")
print(f" - Taxa de aprendizado: {LEARNING_RATE_INIT}")
print(f" - Épocas: {MAX_ITER}")

resultados_q1 = []
seeds = [42, 123, 456, 789, 101112]

for i, seed in enumerate(seeds):
    print(f"\nTreinamento {i+1}/5 (seed = {seed})...")
    start_time = time.time()

    mlp = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation=ACTIVATION,
        solver=SOLVER,
        learning_rate_init=LEARNING_RATE_INIT,
        max_iter=MAX_ITER,
        batch_size=BATCH_SIZE,
        random_state=seed,
        verbose=False,
        early_stopping=False
    )

    mlp.fit(X_train, y_train)

    y_train_pred = mlp.predict(X_train)
    loss_final = mlp.loss_curve_[-1] if mlp.loss_curve_ else 0
    acc_final = accuracy_score(y_train, y_train_pred)

    tempo = time.time() - start_time

    resultados_q1.append({
        "Treinamento": i+1,
        "Seed": seed,
        "Loss Final": loss_final,
        "Acurácia Final": acc_final,
        "Épocas": len(mlp.loss_curve_),
        "Tempo (s)": tempo
    })

    print(f" Loss FInal: {loss_final:.4f} | Acurácia: {acc_final:.4f} | Épocas: {len(mlp.loss_curve_)}")

df_q1 = pd.DataFrame(resultados_q1)
print("\n" + "-"*50)
print("Resultados dos 5 treinamentos:")
print("-"*50)
print(df_q1.to_string(index=False))
print(f"\nMédia da Acurácia: {df_q1['Acurácia Final'].mean():.4f} ± {df_q1['Acurácia Final'].std():.4f}")
print(f"Média da Loss: {df_q1['Loss Final'].mean():.4f} ± {df_q1['Loss Final'].std():.4f}")

# Gráficos da Questão 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Re-treinar para obter curvas completas (uma por seed)
for i, seed in enumerate(seeds):
    mlp = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS, activation=ACTIVATION, solver=SOLVER,
        learning_rate_init=LEARNING_RATE_INIT, max_iter=MAX_ITER, batch_size=BATCH_SIZE,
        random_state=seed, verbose=False
    )
    mlp.fit(X_train, y_train)

    epochs = range(1, len(mlp.loss_curve_) + 1)
    axes[0].plot(epochs, mlp.loss_curve_, label=f"Seed {seed}", linewidth=1.5)

    # Calcular acurácia por época
    acc_curve = []
    for j in range(1, len(mlp.loss_curve_) + 1):
        mlp_partial = MLPClassifier(
            hidden_layer_sizes=HIDDEN_LAYERS, activation=ACTIVATION, solver=SOLVER,
            learning_rate_init=LEARNING_RATE_INIT, max_iter=j, batch_size=BATCH_SIZE,
            random_state=seed, warm_start=True, verbose=False
        )
        mlp_partial.fit(X_train, y_train)
        acc_curve.append(accuracy_score(y_train, mlp_partial.predict(X_train)))
    axes[1].plot(epochs, acc_curve, label=f"Seed {seed}", linewidth=1.5)

axes[0].set_xlabel("Épocas", fontsize=12)
axes[0].set_ylabel("Loss (Cross-Entropy)", fontsize=12)
axes[0].set_title("Curvas de Convergência - Loss", fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Épocas", fontsize=12)
axes[1].set_ylabel("Acurácia", fontsize=12)
axes[1].set_title("Curvas de Convergência - Acurácia", fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("graficos/questao1_convergencia.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n" + "="*60)
print("Questão 2: Exploração de hiperparâmetros")
print("Taxa de Aprendizado e Termo Momento")
print("="*60)

# SGD com momento
LEARNING_RATES = [0.001, 0.005, 0.01, 0.05, 0.1]
MOMENTUMS = [0.5, 0.7, 0.9, 0.95]
CRITERIO_PARADA = 0.01
MAX_EPOCAS = 500

print(f"Valores testados:")
print(f" - Taxa de aprendizado: {LEARNING_RATES}")
print(f" - Momento: {MOMENTUMS}")
print(f" - Critério de parada: Loss <= {CRITERIO_PARADA}")
print(f" - Máximo de épocas: {MAX_EPOCAS}")

resultados_q2 = []

for lr in LEARNING_RATES:
    for mom in MOMENTUMS:
        print(f"\nTestando lr={lr}, momentum={mom}...")

        mlp = MLPClassifier(
            hidden_layer_sizes=HIDDEN_LAYERS,
            activation=ACTIVATION,
            solver="sgd",
            learning_rate_init=lr,
            momentum=mom,
            max_iter=MAX_EPOCAS,
            batch_size=BATCH_SIZE,
            random_state=SEED,
            verbose=False
        )

        start_time = time.time()
        mlp.fit(X_train, y_train)
        tempo = time.time() - start_time

        loss_curve = mlp.loss_curve_
        loss_final = loss_curve[-1] if loss_curve else float("inf")

        # Verificar se atingiu o critério
        convergiu = loss_final <= CRITERIO_PARADA
        epocas_convergencia = None

        if convergiu:
            # Encontrar época em que atingiu o critério
            for epoch, loss in enumerate(loss_curve):
                if loss <= CRITERIO_PARADA:
                    epocas_convergencia = epoch + 1
                    break
        
        resultados_q2.append({
            "Learning Rate": lr,
            "Momentum": mom,
            "Convergência": "Sim" if convergiu else "Não",
            "Épocas": epocas_convergencia if convergiu else MAX_EPOCAS,
            "Loss Final": loss_final,
            "Tempo (s)": round(tempo, 2)
        })

        status = "OK" if convergiu else "DIVERGENTE"
        print(f" {status} | Loss: {loss_final:.4f} | Épocas: {epocas_convergencia if convergiu else MAX_EPOCAS}")

df_q2 = pd.DataFrame(resultados_q2)
print("\n" + "-"*30)
print("Resultados do Grid Search:")
print("-"*30)
print(df_q2.to_string(index=False))

# Gráfico de calor
pivot_epocas = df_q2.pivot(index="Learning Rate", columns="Momentum", values="Épocas")
pivot_convergiu = df_q2.pivot(index="Learning Rate", columns="Momentum", values="Convergência")

fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(pivot_epocas, annot=True, fmt=".0f", cmap="RdYlGn", ax=ax,
            cbar_kws={"label": "Épocas até convergência"})
ax.set_title("Épocas até Convergência (Loss <= 0.01)", fontsize=14)
ax.set_xlabel("Momento")
ax.set_ylabel("Taxa de Aprendizado")
plt.tight_layout()
plt.savefig("graficos/q2_heatmap.png", dpi=150, bbox_inches="tight", format="png")
plt.close()

# Identificar melhor combinação
melhor_q2 = df_q2[df_q2["Convergência"] == "Sim"].nsmallest(1, "Épocas")
if len(melhor_q2) > 0:
    BEST_LR = melhor_q2.iloc[0]["Learning Rate"]
    BEST_MOM = melhor_q2.iloc[0]["Momentum"]
    print(f"Melhor combinação: lr={BEST_LR}, momentum={BEST_MOM} ({melhor_q2.iloc[0]["Épocas"]} épocas)")
else:
    BEST_LR = 0.01
    BEST_MOM = 0.9
    print(f"\n Nenhuma combinação convergiu. Usando padrão: lr={BEST_LR}, momentum={BEST_MOM}")
    

In [ ]:
# Questão 3: Topologia da Rede
print("\n" + "="*60)
print("Questão 3: Topologia da Rede")
print("Influência do número de camadas e neurônios")
print("="*60)

TOPOLOGIAS_TESTAR = [
    (16,), (32,), (64,), (128,),
    (16, 8), (32, 16), (64, 32), (128, 64), (256, 128),
    (32, 16, 8), (64, 32, 16), (128, 64, 32)
]

print(f"Topologias testadas: {len(TOPOLOGIAS_TESTAR)}")
print(f"Hiperparâmetros fixos: lr={BEST_LR}, momentum={BEST_MOM}")

resultados_q3 = []

for topo in TOPOLOGIAS_TESTAR:
    print(f"\nTestando topologia: {topo}...")
    mlp = MLPClassifier(
        hidden_layer_sizes=topo,
        activation=ACTIVATION,
        solver="sgd",
        learning_rate_init=BEST_LR,
        momentum=BEST_MOM,
        max_iter=300,
        batch_size=BATCH_SIZE,
        random_state=SEED,
        verbose=False
    )

    start_time = time.time()
    mlp.fit(X_train, y_train)
    tempo = time.time() - start_time

    # Avaliar em treino e validação
    y_train_pred = mlp.predict(X_train)
    y_val_pred = mlp.predict(X_val)

    loss_treino = mlp.loss_curve_[-1] if mlp.loss_curve_ else 0
    acc_treino = accuracy_score(y_train, y_train_pred)
    acc_val = accuracy_score(y_val, y_val_pred)
    f1_val = f1_score(y_val, y_val_pred, average="weighted")

    # Detectar overfitting/underfitting
    diff = acc_treino - acc_val
    if acc_treino < 0.80:
        status = "Underfitting"
    elif diff > 0.05 and acc_treino > 0.93:
        status = "Overfitting"
    else:
        status = "Ideal"

    resultados_q3.append({
        "Topologia": str(topo),
        "Neurônios": sum(topo),
        "Camadas": len(topo),
        "Loss Treino": round(loss_treino, 4),
        "Acc Treino": round(acc_treino, 4),
        "Acc Val": round(acc_val, 4),
        "F1 Val": round(f1_val, 4),
        "Tempo (s)": round(tempo, 2),
        "Status": status
    })

    print(f" Acc Treino: {acc_treino:.4f} | Acc Val: {acc_val:.4f} | F1: {f1_val:.4f} | Status: {status}")

df_q3 = pd.DataFrame(resultados_q3)
print("\n" + "-"*80)
print("RESULTADOS DAS TOPOLOGIAS:")
print("-"*80)
print(df_q3.to_string(index=False))

# Selecionar as 4 melhores topologias (baseado em F1 na validação)
melhores_q3 = df_q3[df_q3['Status'] == 'Ideal'].nlargest(4, 'F1 Val')
print("\n" + "-"*50)
print("4 MELHORES TOPOLOGIAS SELECIONADAS:")
print("-"*50)
for i, (idx, row) in enumerate(melhores_q3.iterrows()):
    print(f"{i+1}. {row['Topologia']} | F1: {row['F1 Val']:.4f} | Acc Val: {row['Acc Val']:.4f} | Status: {row['Status']}")

TOPOLOGIAS_SELECIONADAS = melhores_q3['Topologia'].tolist()

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de loss vs complexidade
axes[0].scatter(df_q3['Neurônios'], df_q3['Acc Treino'], label='Treino', alpha=0.6, s=50)
axes[0].scatter(df_q3['Neurônios'], df_q3['Acc Val'], label='Validação', alpha=0.6, s=50)
axes[0].set_xlabel('Número de Neurônios', fontsize=12)
axes[0].set_ylabel('Acurácia', fontsize=12)
axes[0].set_title('Acurácia vs Complexidade da Rede', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico de barras das melhores
melhores_plot = df_q3.nlargest(8, 'F1 Val')
sns.barplot(data=melhores_plot, x='Topologia', y='F1 Val', hue='Status', ax=axes[1])
axes[1].set_xlabel('Topologia', fontsize=12)
axes[1].set_ylabel('F1-Score', fontsize=12)
axes[1].set_title('Topologias com Melhor F1-Score', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('graficos/q3_topology_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Versão corrigida da Questão 4
print("\n" + "="*60)
print("Questão 4: Influência da Quantidade de Dados")
print("="*60)

# Usar a melhor topologia da questão 3
BEST_TOPOLOGY = TOPOLOGIAS_SELECIONADAS[0]
print(f"Usando topologia: {BEST_TOPOLOGY}")

TAMANHOS = [0.10, 0.20, 0.40, 0.60, 0.80, 1.00]
resultados_q4 = []

for frac in TAMANHOS:
    n_samples = int(len(X_train) * frac)
    print(f"\nTestando com {frac*100:.0f}% dos dados ({n_samples} amostras)...")
    
    # CORREÇÃO: tratar o caso 100% separadamente
    if frac == 1.0:
        # Usar todos os dados de treino
        X_train_frac = X_train
        y_train_frac = y_train
    else:
        # Amostragem estratificada para frações < 1
        sss = StratifiedShuffleSplit(n_splits=1, train_size=frac, random_state=SEED)
        train_idx, _ = next(sss.split(X_train, y_train))
        X_train_frac = X_train[train_idx]
        y_train_frac = y_train[train_idx]
    
    mlp = MLPClassifier(
        hidden_layer_sizes=eval(BEST_TOPOLOGY) if isinstance(BEST_TOPOLOGY, str) else BEST_TOPOLOGY,
        activation=ACTIVATION,
        solver="sgd",
        learning_rate_init=BEST_LR,
        momentum=BEST_MOM,
        max_iter=300,
        batch_size=BATCH_SIZE,
        random_state=SEED,
        verbose=False
    )
    
    start_time = time.time()
    mlp.fit(X_train_frac, y_train_frac)
    tempo = time.time() - start_time
    
    # Avaliar
    y_train_pred = mlp.predict(X_train_frac)
    y_val_pred = mlp.predict(X_val)
    
    acc_train = accuracy_score(y_train_frac, y_train_pred)
    acc_val = accuracy_score(y_val, y_val_pred)
    f1_val = f1_score(y_val, y_val_pred, average="weighted")
    
    resultados_q4.append({
        "Fração": frac,
        "N Amostras": n_samples,
        "Acc Treino": round(acc_train, 4),
        "Acc Val": round(acc_val, 4),
        "F1 Val": round(f1_val, 4),
        "Tempo (s)": round(tempo, 2)
    })
    
    print(f"  Acc Treino: {acc_train:.4f} | Acc Val: {acc_val:.4f} | F1: {f1_val:.4f} | Tempo: {tempo:.2f}s")

df_q4 = pd.DataFrame(resultados_q4)
print("\n" + "-"*60)
print("Resultados - Influência do Tamanho dos Dados:")
print("-"*60)
print(df_q4.to_string(index=False))

# Curvas de generalização
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(df_q4["Fração"] * 100, df_q4["Acc Val"], "o-", linewidth=2, markersize=8, label="Acurácia Validação")
ax.plot(df_q4["Fração"] * 100, df_q4["F1 Val"], "s-", linewidth=2, markersize=8, label="F1-Score")
ax.set_xlabel("Percentual de Dados de Treinamento (%)", fontsize=12)
ax.set_ylabel("Desempenho", fontsize=12)
ax.set_title("Curva de Generalização - Impacto da Quantidade de Dados", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("graficos/q4_generalization_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Questão 5: Influência dos Atributos
print("\n" + "="*60)
print("QUESTÃO 5: INFLUÊNCIA DOS ATRIBUTOS")
print("="*60)

# Nomes dos atributos
feature_names = df.drop('Class', axis=1).columns.tolist()
print(f"Atributos disponíveis: {len(feature_names)}")
print(feature_names)

# Estratégias de seleção de atributos
print("\nEstratégias de seleção:")
print("1. Todos os atributos (baseline)")
print("2. Redução PCA para 8 componentes")
print("3. Seleção baseada em correlação (remover redundantes)")
print("4. Top 8 atributos por variância")

# Calcular correlação com target (para atributos contínuos, usamos ANOVA-like)
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=8)
selector.fit(X_train, y_train)
top8_idx = selector.get_support(indices=True)
top8_names = [feature_names[i] for i in top8_idx]
print(f"\nTop 8 atributos por importância estatística: {top8_names}")

# PCA
pca = PCA(n_components=8)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# Matriz de correlação
corr_matrix = df[feature_names].corr()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j]))
print(f"\nPares com correlação > 0.9: {len(high_corr_pairs)}")

# Remover atributos redundantes (exemplo: manter apenas um de cada par)
redundantes = set()
for pair in high_corr_pairs:
    redundantes.add(pair[1])
keep_cols = [col for col in feature_names if col not in redundantes]
keep_idx = [feature_names.index(col) for col in keep_cols]
print(f"Atributos mantidos após remover redundantes ({len(keep_cols)}): {keep_cols}")

resultados_q5 = []
ESTRATEGIAS = [
    ('Todos os atributos', X_train, X_val, feature_names),
    ('PCA (8 componentes)', X_train_pca, X_val_pca, [f'PC{i+1}' for i in range(8)]),
    ('Top 8 atributos (ANOVA)', X_train[:, top8_idx], X_val[:, top8_idx], top8_names),
    ('Sem atributos redundantes', X_train[:, keep_idx], X_val[:, keep_idx], keep_cols)
]

for nome, X_tr, X_vl, cols in ESTRATEGIAS:
    print(f"\nTestando estratégia: {nome}")
    print(f"  {X_tr.shape[1]} atributos")
    
    mlp = MLPClassifier(
        hidden_layer_sizes=eval(BEST_TOPOLOGY) if isinstance(BEST_TOPOLOGY, str) else BEST_TOPOLOGY,
        activation=ACTIVATION,
        solver='sgd',
        learning_rate_init=BEST_LR,
        momentum=BEST_MOM,
        max_iter=200,
        batch_size=BATCH_SIZE,
        random_state=42,
        verbose=False
    )
    
    start_time = time.time()
    mlp.fit(X_tr, y_train)
    tempo = time.time() - start_time
    
    y_train_pred = mlp.predict(X_tr)
    y_val_pred = mlp.predict(X_vl)
    
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_val = accuracy_score(y_val, y_val_pred)
    f1_val = f1_score(y_val, y_val_pred, average='weighted')
    
    resultados_q5.append({
        'Estratégia': nome,
        'Atributos': X_tr.shape[1],
        'Acc Treino': round(acc_train, 4),
        'Acc Val': round(acc_val, 4),
        'F1 Val': round(f1_val, 4),
        'Tempo (s)': round(tempo, 2)
    })
    
    print(f"  Acc Val: {acc_val:.4f} | F1: {f1_val:.4f} | Tempo: {tempo:.2f}s")

df_q5 = pd.DataFrame(resultados_q5)
print("\n" + "-"*60)
print("RESULTADOS - INFLUÊNCIA DOS ATRIBUTOS:")
print("-"*60)
print(df_q5.to_string(index=False))

# Identificar melhor estratégia
melhor_q5 = df_q5.loc[df_q5['F1 Val'].idxmax()]
print(f"\nMelhor estratégia: {melhor_q5['Estratégia']} (F1: {melhor_q5['F1 Val']:.4f})")

In [ ]:
# Questão 6: Validação Inicial (Conjunto de Teste)
print("\n" + "="*60)
print("QUESTÃO 6: VALIDAÇÃO INICIAL")
print("Comparação das 4 melhores configurações no conjunto de teste")
print("="*60)

# Reconstruir as 4 melhores topologias com os hiperparâmetros otimizados
resultados_q6 = []

for i, topo_str in enumerate(TOPOLOGIAS_SELECIONADAS):
    topo = eval(topo_str) if isinstance(topo_str, str) else topo_str
    print(f"\nTestando configuração {i+1}: {topo}")
    
    mlp = MLPClassifier(
        hidden_layer_sizes=topo,
        activation=ACTIVATION,
        solver='sgd',
        learning_rate_init=BEST_LR,
        momentum=BEST_MOM,
        max_iter=300,
        batch_size=BATCH_SIZE,
        random_state=42,
        verbose=False
    )
    
    start_time = time.time()
    mlp.fit(X_train, y_train)
    tempo_treino = time.time() - start_time
    
    # Avaliar em treino e teste
    y_train_pred = mlp.predict(X_train)
    y_test_pred = mlp.predict(X_test)
    
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred, average='weighted')
    precision_test = precision_score(y_test, y_test_pred, average='weighted')
    recall_test = recall_score(y_test, y_test_pred, average='weighted')
    
    resultados_q6.append({
        'Configuração': i+1,
        'Topologia': str(topo),
        'Acc Treino': round(acc_train, 4),
        'Acc Teste': round(acc_test, 4),
        'F1 Teste': round(f1_test, 4),
        'Precisão Teste': round(precision_test, 4),
        'Recall Teste': round(recall_test, 4),
        'Tempo Treino (s)': round(tempo_treino, 2)
    })
    
    print(f"  Acc Treino: {acc_train:.4f} | Acc Teste: {acc_test:.4f} | F1: {f1_test:.4f}")

df_q6 = pd.DataFrame(resultados_q6)
print("\n" + "-"*70)
print("RESULTADOS DA VALIDAÇÃO INICIAL (TREINO vs TESTE):")
print("-"*70)
print(df_q6.to_string(index=False))

# Identificar melhor configuração final
melhor_config = df_q6.loc[df_q6['F1 Teste'].idxmax()]
FINAL_TOPOLOGY = eval(melhor_config['Topologia']) if isinstance(melhor_config['Topologia'], str) else melhor_config['Topologia']
print(f"\nMELHOR CONFIGURAÇÃO FINAL:")
print(f"   Topologia: {FINAL_TOPOLOGY}")
print(f"   Acurácia Teste: {melhor_config['Acc Teste']:.4f}")
print(f"   F1-Score Teste: {melhor_config['F1 Teste']:.4f}")

# Curvas de validação para a melhor configuração
print("\nGerando curvas de validação para a melhor configuração...")

mlp_final = MLPClassifier(
    hidden_layer_sizes=FINAL_TOPOLOGY,
    activation=ACTIVATION,
    solver='sgd',
    learning_rate_init=BEST_LR,
    momentum=BEST_MOM,
    max_iter=300,
    batch_size=BATCH_SIZE,
    random_state=42,
    verbose=False
)

mlp_final.fit(X_train, y_train)

# Plotar curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(mlp_final.loss_curve_)+1), mlp_final.loss_curve_, 'b-', linewidth=2)
axes[0].set_xlabel('Épocas', fontsize=12)
axes[0].set_ylabel('Loss (Cross-Entropy)', fontsize=12)
axes[0].set_title('Curva de Perda - Melhor Configuração', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Calcular acurácia ao longo do treinamento
acc_curve = []
for epoch in range(1, len(mlp_final.loss_curve_)+1, 10):
    mlp_partial = MLPClassifier(
        hidden_layer_sizes=FINAL_TOPOLOGY, activation=ACTIVATION, solver='sgd',
        learning_rate_init=BEST_LR, momentum=BEST_MOM, max_iter=epoch,
        batch_size=BATCH_SIZE, random_state=42, warm_start=True, verbose=False
    )
    mlp_partial.fit(X_train, y_train)
    acc_curve.append((epoch, accuracy_score(y_train, mlp_partial.predict(X_train))))

epochs_acc, acc_values = zip(*acc_curve)
axes[1].plot(epochs_acc, acc_values, 'r-', linewidth=2)
axes[1].set_xlabel('Épocas', fontsize=12)
axes[1].set_ylabel('Acurácia', fontsize=12)
axes[1].set_title('Curva de Acurácia - Melhor Configuração', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q6_validation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Matriz de confusão para o teste
from sklearn.metrics import ConfusionMatrixDisplay

y_test_pred_final = mlp_final.predict(X_test)
cm = confusion_matrix(y_test, y_test_pred_final)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Matriz de Confusão - Teste (Acurácia: {melhor_config["Acc Teste"]:.4f})', fontsize=14)
plt.tight_layout()
plt.savefig('graficos/q6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Relatório de classificação
print("\n" + "="*60)
print("RELATÓRIO DE CLASSIFICAÇÃO - MELHOR MODELO")
print("="*60)
print(classification_report(y_test, y_test_pred_final, target_names=classes))

In [ ]:
# Questão 7: Validação Cruzada (K-Fold)
print("\n" + "="*60)
print("QUESTÃO 7: VALIDAÇÃO CRUZADA (K-FOLD)")
print("Avaliação consolidada do modelo final")
print("="*60)

K_FOLDS = 5
print(f"Utilizando k = {K_FOLDS} folds (estratificados)")

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

resultados_cv = {
    'fold': [],
    'acc_treino': [],
    'acc_teste': [],
    'f1_teste': [],
    'precision_teste': [],
    'recall_teste': [],
    'loss_treino': []
}

fold = 1
for train_idx, test_idx in skf.split(X_scaled, y_encoded):
    print(f"\nFold {fold}/{K_FOLDS}...")
    
    X_tr_fold = X_scaled[train_idx]
    X_te_fold = X_scaled[test_idx]
    y_tr_fold = y_encoded[train_idx]
    y_te_fold = y_encoded[test_idx]
    
    mlp = MLPClassifier(
        hidden_layer_sizes=FINAL_TOPOLOGY,
        activation=ACTIVATION,
        solver='sgd',
        learning_rate_init=BEST_LR,
        momentum=BEST_MOM,
        max_iter=300,
        batch_size=BATCH_SIZE,
        random_state=42,
        verbose=False
    )
    
    mlp.fit(X_tr_fold, y_tr_fold)
    
    y_tr_pred = mlp.predict(X_tr_fold)
    y_te_pred = mlp.predict(X_te_fold)
    
    resultados_cv['fold'].append(fold)
    resultados_cv['acc_treino'].append(accuracy_score(y_tr_fold, y_tr_pred))
    resultados_cv['acc_teste'].append(accuracy_score(y_te_fold, y_te_pred))
    resultados_cv['f1_teste'].append(f1_score(y_te_fold, y_te_pred, average='weighted'))
    resultados_cv['precision_teste'].append(precision_score(y_te_fold, y_te_pred, average='weighted'))
    resultados_cv['recall_teste'].append(recall_score(y_te_fold, y_te_pred, average='weighted'))
    resultados_cv['loss_treino'].append(mlp.loss_curve_[-1] if mlp.loss_curve_ else 0)
    
    print(f"  Acc Treino: {resultados_cv['acc_treino'][-1]:.4f}")
    print(f"  Acc Teste: {resultados_cv['acc_teste'][-1]:.4f}")
    print(f"  F1 Teste: {resultados_cv['f1_teste'][-1]:.4f}")
    
    fold += 1

df_cv = pd.DataFrame(resultados_cv)

print("\n" + "-"*60)
print("RESULTADOS DA VALIDAÇÃO CRUZADA (5-FOLD):")
print("-"*60)
print(df_cv.to_string(index=False))

print("\n" + "-"*60)
print("ESTATÍSTICAS CONSIDERANDO OS K FOLDS:")
print("-"*60)
print(f"Acurácia Treino: {np.mean(df_cv['acc_treino']):.4f} ± {np.std(df_cv['acc_treino']):.4f}")
print(f"Acurácia Teste:  {np.mean(df_cv['acc_teste']):.4f} ± {np.std(df_cv['acc_teste']):.4f}")
print(f"F1-Score Teste:  {np.mean(df_cv['f1_teste']):.4f} ± {np.std(df_cv['f1_teste']):.4f}")
print(f"Precisão Teste:  {np.mean(df_cv['precision_teste']):.4f} ± {np.std(df_cv['precision_teste']):.4f}")
print(f"Recall Teste:    {np.mean(df_cv['recall_teste']):.4f} ± {np.std(df_cv['recall_teste']):.4f}")

# Boxplot da validação cruzada
fig, ax = plt.subplots(figsize=(10, 6))
df_cv_melted = pd.melt(df_cv[['acc_teste', 'f1_teste', 'precision_teste', 'recall_teste']], 
                       var_name='Métrica', value_name='Valor')
sns.boxplot(data=df_cv_melted, x='Métrica', y='Valor', ax=ax)
ax.set_ylim(0, 1)
ax.set_title(f'Distribuição das Métricas - Validação Cruzada ({K_FOLDS}-fold)', fontsize=14)
ax.set_ylabel('Valor', fontsize=12)
ax.set_xlabel('Métrica', fontsize=12)
plt.tight_layout()
plt.savefig('graficos/q7_cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Resumo Final 
print("\n" + "="*80)
print("RESUMO FINAL DO TRABALHO")
print("="*80)

print("\nCONFIGURAÇÕES FINAIS:")
print(f"  - Melhor Topologia: {FINAL_TOPOLOGY}")
print(f"  - Taxa de Aprendizado: {BEST_LR}")
print(f"  - Momento: {BEST_MOM}")
print(f"  - Função Ativação: ReLU (ocultas) / Softmax (saída)")
print(f"  - Otimizador: SGD com momento")

print("\nDESEMPENHO DO MODELO FINAL:")
print(f"  - Acurácia (Treino): {resultados_q6[melhor_config.name]['Acc Treino']:.4f}")
print(f"  - Acurácia (Teste):  {melhor_config['Acc Teste']:.4f}")
print(f"  - F1-Score (Teste):  {melhor_config['F1 Teste']:.4f}")

print("\nVALIDAÇÃO CRUZADA (5-fold):")
print(f"  - Acurácia Média: {np.mean(df_cv['acc_teste']):.4f} ± {np.std(df_cv['acc_teste']):.4f}")
print(f"  - F1-Score Médio: {np.mean(df_cv['f1_teste']):.4f} ± {np.std(df_cv['f1_teste']):.4f}")

print("\nTRABALHO CONCLUÍDO COM SUCESSO!")
print("Todos os gráficos e resultados foram salvos no diretório atual.")